# Guide 0 — How Everything Connects (read this first!) 🗺️

> **PYNQ Bootcamp guide.** This notebook takes one piece of the big competition program and explains it in small steps. Almost all of the code here is the *real* code that runs during a match — we've just split it up and added plain-English notes so it's easy to follow. (The one exception is the *Matching Strategy* guide, where the game plan is written as pseudocode for you to think through.)

## The big picture

The full competition program is large, so we split it into 12 smaller guides. This
overview guide shows how all the pieces fit together, like the picture on the front
of a puzzle box. **Read this first**, then dive into the other guides.

Think of the program as a team of workers:

- The **camera + YOLO eye** (Guides 2–5) look at the board and read cards.
- The **corner stickers + grid math** (Guides 3–4) figure out where each square is.
- The **referee translator** (Guide 7) sends and receives messages from the game.
- The **strategy brain** (Guide 8) decides which cards to flip.
- The **control panel** (Guide 10) is the buttons you click.
- The **helpers** (Guides 6, 9, 11, 12) handle riddles, hints, and the robot arm.


## What happens during one turn

Here's the whole story of a single turn, and which guide handles each part:

1. The referee sends a message: *"It's your turn."* → **Guide 7** receives it.
2. The **strategy brain** picks two squares to flip. → **Guide 8**
3. The program tells the referee *"flip these two."* → **Guide 7**
4. A human flips the real cards. The referee says *"a card was revealed at C4."*
5. The **camera reads that one square** and YOLO says what it is. → **Guides 2–5**
6. After both cards are read, the brain decides *"match or no match?"* and reports
   it. → **Guide 8 + Guide 7**
7. The **control panel** updates the scoreboard on screen. → **Guide 10**

All of this repeats, by itself, every turn.


## The "engine" that runs it all

Underneath, there's one small loop that keeps the whole thing going. It just checks
for new messages from the referee over and over, and hands each message to the right
helper. This is the real code — and it's surprisingly short for something that runs
the entire match!

`self.client.poll()` gets new messages (Guide 7), and `_handle_message` sends each
one to the correct piece (the strategy brain, the card reader, the hint helpers, and
so on).


In [ ]:
    def _run_loop(self):
        while not self._stop_event.is_set():
            if self._pause_event.is_set():
                time.sleep(self.POLL_INTERVAL_SECONDS)
                continue
            try:
                for message in self.client.poll():
                    self._handle_message(message)
            except Exception as exc:
                print(f'[match] poll error: {type(exc).__name__}: {exc}')
            time.sleep(self.POLL_INTERVAL_SECONDS)

## The map of guides

Follow the arrows: a guide's arrow points to the guides it *uses*.

```
Guide 1  (Setup & settings)          ← everything needs this first
   │
Guide 2  (Camera + YOLO eye)
   │
Guide 3  (Grid math)  ←── Guide 4 (Corner stickers)
   │                          │
Guide 5  (Read one card)  ←───┘   uses Guides 2, 3, 4
   │
Guide 8  (Strategy brain) ★ the pseudocode one
   │
Guide 7  (Referee translator)     the brain talks through this
   │
Guide 10 (Control panel)          shows it all on screen

Helpers used along the way:
   Guide 6  (Riddle solver)     — pre-game
   Guide 11 (Number-hint reader / MNIST)
   Guide 12 (Free hints & riddle messages)
   Guide 9  (Robot arm — decoration only)
```

## Suggested reading order

For total beginners, read them in this order: **0 → 1 → 2 → 3 → 4 → 5 → 7 → 8 →
10**, then the helper guides **6, 11, 12, 9** whenever you're curious. Guides 1–5
build on each other; the rest can be read in any order once you've done those.
